In [ ]:
import requests
import pandas as pd

In [ ]:
# Effectuer la requête GET pour obtenir la liste des tags
FINK_API = "https://api.lsst.fink-portal.org"
TAGS_API = "https://api.lsst.fink-portal.org/api/v1/tags"
response = requests.get(TAGS_API)

if response.status_code == 200:
    # La réponse est généralement une liste JSON des noms de tags
    tags = response.json()
    print(f"Nombre de tags trouvés : {len(tags)}")
    print("Liste des tags :")
    for tag in tags.items():
        print(f"- {tag}")
else:
    print(f"Erreur lors de la requête : {response.status_code}")

In [ ]:
def fetch_tags() -> pd.DataFrame:
    """
    Fetch the Fink classification tags from /api/v1/tags.

    Tags represent event-level classifications (SN Ia, kilonova, AGN flare, etc.).
    NOTE: tags are retrieved for information only — objects are NOT filtered by tag.

    Same POST-based approach as fetch_blocks(): uses POST with
    'output-format': 'json' and handles all response shapes gracefully.

    Returns an empty DataFrame if the endpoint is unavailable or returns
    no data, with a descriptive message for each failure mode.
    """
    url = f"{FINK_API}/api/v1/tags"
    # payload = {"output-format": "json"}
    try:
        # r = requests.post(url, json=payload, timeout=30)
        r = requests.get(url)

        # HTTP 404 / 405: endpoint does not exist or method not allowed
        if r.status_code in (404, 405):
            print(f"fetch_tags: endpoint not available (HTTP {r.status_code})")
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            print("fetch_tags: empty response from API")
            return pd.DataFrame()
        # Normalise response shape to a DataFrame
        if isinstance(data, list):
            if len(data) == 0:
                return pd.DataFrame()
            if isinstance(data[0], dict):
                return pd.DataFrame(data)  # standard list-of-dicts
            else:
                return pd.DataFrame({"name": data})  # flat list of strings/scalars
        if isinstance(data, dict):
            return pd.DataFrame([data])  # single-row dict
        # Unknown shape — wrap as best-effort
        return pd.DataFrame({"raw": [data]})
    except requests.exceptions.HTTPError as e:
        print(f"fetch_tags HTTP error: {e}")
        return pd.DataFrame()
    except requests.exceptions.ConnectionError as e:
        print(f"fetch_tags connection error: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"fetch_tags unexpected error: {type(e).__name__}: {e}")
        return pd.DataFrame()


print("API helpers defined.")

In [ ]:
fetch_tags().T

In [ ]:
SCHEMA_API = "https://api.lsst.fink-portal.org/api/v1/schema"

payload = {"endpoint": "/api/v1/tags"}

res_schema = requests.post(SCHEMA_API, json=payload)


if response.status_code == 200:
    schema = response.json()
    print("Schéma pour les tags récupéré avec succès :")
    for field, description in list(schema.items())[:5]:
        print(f"- {field}: {description}")
else:
    print(f"Erreur {response.status_code}")
    print(f"Détails : {response.text}")

In [ ]:
def fetch_blocks() -> pd.DataFrame:
    """
    Fetch the Fink block definitions from /api/v1/blocks.

    The Fink LSST API uses POST for all endpoints, including those that
    take no mandatory parameters. The 'output-format': 'json' key is
    required to ensure a JSON response rather than CSV or Avro.

    The response may be:
      - a list of dicts  → pd.DataFrame(data)
      - a list of strings / scalars  → pd.DataFrame({'name': data})
      - a plain dict  → pd.DataFrame([data])

    Returns an empty DataFrame if the endpoint is unavailable or returns
    no data, with a descriptive message for each failure mode.
    """
    url = f"{FINK_API}/api/v1/blocks"
    # payload = {"output-format": "json"}
    try:
        # r = requests.post(url, json=payload, timeout=30)
        r = requests.get(url)
        # HTTP 404 / 405: endpoint does not exist or method not allowed
        if r.status_code in (404, 405):
            print(f"fetch_blocks: endpoint not available (HTTP {r.status_code})")
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            print("fetch_blocks: empty response from API")
            return pd.DataFrame()
        # Normalise response shape to a DataFrame
        if isinstance(data, list):
            if len(data) == 0:
                return pd.DataFrame()
            if isinstance(data[0], dict):
                return pd.DataFrame(data)  # standard list-of-dicts
            else:
                return pd.DataFrame({"name": data})  # flat list of strings/scalars
        if isinstance(data, dict):
            return pd.DataFrame([data])  # single-row dict
        # Unknown shape — wrap as best-effort
        return pd.DataFrame({"raw": [data]})
    except requests.exceptions.HTTPError as e:
        print(f"fetch_blocks HTTP error: {e}")
        return pd.DataFrame()
    except requests.exceptions.ConnectionError as e:
        print(f"fetch_blocks connection error: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"fetch_blocks unexpected error: {type(e).__name__}: {e}")
        return pd.DataFrame()

In [ ]:
fetch_blocks().T